# Notebook 1 — 2D GNN Baseline on QM9

**Project:** Equivariant Graph Neural Networks with Chirality Encoding for Structure-Sensitive Molecular Property Prediction

**Goal of this notebook:** build the simplest possible baseline — atoms as nodes, bonds as edges,
plain message passing, global pooling, one regression head — using **only 2D graph topology**
(no 3D coordinates at all). This gives us a number to beat in every later notebook
(3D EGNN → 3D+Stereo → multi-task Tox21/BBBP).

We deliberately keep this simple and well-instrumented (fixed seed, saved train/val/test split,
saved normalization stats, saved results as JSON) so that Notebooks 2–5 can be evaluated on
*exactly* the same molecules for a fair comparison.


In [ ]:
!pip install -q torch_geometric
!pip install -q rdkit


In [ ]:
import os
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import QM9
from torch_geometric.loader import DataLoader
from torch_geometric.nn import NNConv, global_mean_pool

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)


## 1. Load QM9

QM9 (via PyTorch Geometric) ships ~130k small organic molecules with 19 quantum-mechanical
properties each, plus 3D equilibrium geometry. For this notebook we **ignore** the 3D geometry
(`data.pos`) entirely — that's the whole point of a 2D baseline.


In [ ]:
DATA_ROOT = './data/QM9'
dataset = QM9(root=DATA_ROOT)
print(f"Total molecules in QM9: {len(dataset)}")
print(dataset[0])


## 2. Choose a target property

QM9's `data.y` is a `(1, 19)` tensor. We predict the **HOMO-LUMO gap** (`gap`, index 4), a
standard benchmark target used across most QM9 GNN papers, so our numbers are easy to compare
against published baselines later.


In [ ]:
TARGET_NAMES = [
    'mu', 'alpha', 'homo', 'lumo', 'gap', 'r2', 'zpve', 'u0', 'u298',
    'h298', 'g298', 'cv', 'u0_atom', 'u298_atom', 'h298_atom',
    'g298_atom', 'A', 'B', 'C'
]
TARGET_IDX = 4  # 'gap' (HOMO-LUMO gap), a commonly benchmarked property
print(f"Predicting target: {TARGET_NAMES[TARGET_IDX]} (index {TARGET_IDX})")


## 3. Train / val / test split

Using the full ~130k molecules is unnecessary for a class project and slows down every later
notebook (especially the RDKit conformer-generation step in Notebook 2). We subsample
**20,000 molecules** with a fixed seed and split 80/10/10.

We save the resulting indices to `qm9_split.pt` so Notebooks 2–5 can load the *same* split and
give a fair, apples-to-apples comparison.


In [ ]:
NUM_SAMPLES = 20000

perm = torch.randperm(len(dataset), generator=torch.Generator().manual_seed(SEED))
subset_indices = perm[:NUM_SAMPLES]

n_train = int(0.8 * NUM_SAMPLES)
n_val = int(0.1 * NUM_SAMPLES)

train_idx = subset_indices[:n_train]
val_idx = subset_indices[n_train:n_train + n_val]
test_idx = subset_indices[n_train + n_val:]

print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

# Save split so later notebooks (3D EGNN, stereo, Tox21, BBBP) can reuse the
# exact same molecules for a fair, apples-to-apples comparison.
torch.save(
    {'train_idx': train_idx, 'val_idx': val_idx, 'test_idx': test_idx,
     'target_idx': TARGET_IDX, 'seed': SEED, 'num_samples': NUM_SAMPLES},
    'qm9_split.pt'
)


## 4. Normalize the target

GNN regression trains much better on a normalized target. We compute mean/std **from the
training set only** (never touch val/test statistics) and save them so later notebooks report
comparable, unnormalized MAE in the same physical units.


In [ ]:
train_targets = torch.stack([dataset[i].y[0, TARGET_IDX] for i in train_idx])
target_mean = train_targets.mean()
target_std = train_targets.std()
print(f"Target mean: {target_mean:.4f}, std: {target_std:.4f}")

torch.save({'mean': target_mean, 'std': target_std}, 'target_norm_stats.pt')


## 5. Dataset wrapper (2D-only)

This wrapper selects+normalizes the single regression target and **explicitly deletes `pos`**
so it's impossible for the model to accidentally see 3D coordinates.


In [ ]:
class QM9TargetWrapper(torch.utils.data.Dataset):
    """Wraps QM9 to expose a single normalized regression target and drop 3D
    coordinates, so this baseline only ever sees 2D graph topology
    (atom features as nodes, bond features as edges)."""

    def __init__(self, base_dataset, indices, target_idx, mean, std):
        self.base = base_dataset
        self.indices = indices
        self.target_idx = target_idx
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        data = self.base[self.indices[i]].clone()
        y = (data.y[0, self.target_idx] - self.mean) / self.std
        data.y = y.view(1)
        if hasattr(data, 'pos'):
            del data.pos  # 2D baseline: no 3D geometry allowed
        return data


train_set = QM9TargetWrapper(dataset, train_idx, TARGET_IDX, target_mean, target_std)
val_set = QM9TargetWrapper(dataset, val_idx, TARGET_IDX, target_mean, target_std)
test_set = QM9TargetWrapper(dataset, test_idx, TARGET_IDX, target_mean, target_std)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
val_loader = DataLoader(val_set, batch_size=256, shuffle=False)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False)


## 6. Model — plain 2D message-passing GNN

- **Nodes** = atoms, described by the atom-feature vector PyG's QM9 loader already provides
  (atomic number one-hot, aromaticity, hybridization, etc. — all derived from 2D structure).
- **Edges** = bonds, described by a bond-type one-hot (`edge_attr`).
- Message passing via `NNConv` (edge-conditioned convolution), global mean pooling, then an
  MLP regression head. No coordinates, no distances, anywhere.


In [ ]:
class GNNBaseline2D(nn.Module):
    """Simple 2D message-passing GNN baseline: atoms as nodes, bonds as
    edges, edge-conditioned message passing, global mean pooling, MLP head.
    Uses no 3D information whatsoever."""

    def __init__(self, node_in_dim, edge_in_dim, hidden_dim=64, num_layers=3):
        super().__init__()
        self.lin_in = nn.Linear(node_in_dim, hidden_dim)

        self.convs = nn.ModuleList()
        for _ in range(num_layers):
            edge_mlp = nn.Sequential(
                nn.Linear(edge_in_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim * hidden_dim),
            )
            self.convs.append(NNConv(hidden_dim, hidden_dim, edge_mlp, aggr='mean'))

        self.norms = nn.ModuleList([nn.BatchNorm1d(hidden_dim) for _ in range(num_layers)])

        self.readout = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        h = F.relu(self.lin_in(x))
        for conv, norm in zip(self.convs, self.norms):
            h = F.relu(norm(conv(h, edge_index, edge_attr)))
        graph_embedding = global_mean_pool(h, batch)
        out = self.readout(graph_embedding)
        return out.view(-1)


In [ ]:
sample = dataset[0]
node_in_dim = sample.x.size(-1)
edge_in_dim = sample.edge_attr.size(-1)
print("Node feature dim:", node_in_dim, "| Edge feature dim:", edge_in_dim)

model = GNNBaseline2D(node_in_dim, edge_in_dim).to(device)
print(model)


## 7. Training utilities


In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()
        pred = model(data)
        loss = F.l1_loss(pred, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, device, mean, std):
    model.eval()
    total_mae_norm = 0.0
    total_mae_real = 0.0
    for data in loader:
        data = data.to(device)
        pred = model(data)
        total_mae_norm += F.l1_loss(pred, data.y, reduction='sum').item()
        pred_real = pred * std.to(device) + mean.to(device)
        y_real = data.y * std.to(device) + mean.to(device)
        total_mae_real += F.l1_loss(pred_real, y_real, reduction='sum').item()
    n = len(loader.dataset)
    return total_mae_norm / n, total_mae_real / n


## 8. Train


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

N_EPOCHS = 60
best_val_mae = float('inf')
history = {'train_loss': [], 'val_mae_norm': [], 'val_mae_real': []}

for epoch in range(1, N_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    val_mae_norm, val_mae_real = evaluate(model, val_loader, device, target_mean, target_std)
    scheduler.step(val_mae_norm)

    history['train_loss'].append(train_loss)
    history['val_mae_norm'].append(val_mae_norm)
    history['val_mae_real'].append(val_mae_real)

    if val_mae_norm < best_val_mae:
        best_val_mae = val_mae_norm
        torch.save(model.state_dict(), 'gnn_2d_baseline_best.pt')

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} | Train L1 (norm): {train_loss:.4f} | "
              f"Val MAE (norm): {val_mae_norm:.4f} | Val MAE (real units): {val_mae_real:.4f}")


## 9. Test evaluation


In [ ]:
model.load_state_dict(torch.load('gnn_2d_baseline_best.pt'))
test_mae_norm, test_mae_real = evaluate(model, test_loader, device, target_mean, target_std)
print(f"Test MAE (normalized): {test_mae_norm:.4f}")
print(f"Test MAE (real units, {TARGET_NAMES[TARGET_IDX]}): {test_mae_real:.4f}")


## 10. Save results

Saved as JSON so Notebook 6 (final comparison) — and Notebook 2 right away — can load this
number without re-running training.


In [ ]:
results = {
    'model_name': '2D_GNN_baseline',
    'target': TARGET_NAMES[TARGET_IDX],
    'target_idx': TARGET_IDX,
    'num_samples_used': NUM_SAMPLES,
    'num_train': len(train_idx),
    'num_val': len(val_idx),
    'num_test': len(test_idx),
    'test_mae_normalized': test_mae_norm,
    'test_mae_real_units': test_mae_real,
}

with open('results_2d_baseline.json', 'w') as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))


## 11. Training curve


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.plot(history['train_loss'], label='Train L1 loss (normalized)')
plt.plot(history['val_mae_norm'], label='Val MAE (normalized)')
plt.xlabel('Epoch')
plt.ylabel('Loss / MAE')
plt.title('2D GNN Baseline — Training Curve')
plt.legend()
plt.show()


## Done — files to keep for later notebooks

Download / keep these three files (from the Colab file browser, or push them to Google Drive):

- `qm9_split.pt` — the train/val/test molecule indices
- `target_norm_stats.pt` — target mean/std used for normalization
- `results_2d_baseline.json` — this baseline's test MAE, for the final comparison table

**Next up (Notebook 2):** add 3D geometry via RDKit-generated conformers and an Equivariant
GNN (EGNN), and see how much the HOMO-LUMO gap MAE improves over this 2D baseline.
